In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

# ============================================================
# CREATE SEASONAL CASE-STUDY FILES FOR IDEAL UI/API EVALUATION
# ============================================================
#
# For each selected seasonal case:
#   - creates:
#       C:/IDEAL_Programming/New_Evaluation/SEASONAL/<season>_<home_id>_<target_date>/
#
#   - saves UI-ready history_store.csv with columns:
#       timestamp, consumption_Wh
#
#   - saves actual_target_day.csv for later evaluation
#   - saves static_metadata.csv / static_metadata.json
#   - saves case_metadata.json
#   - saves global summary files
#
# Important:
#   history window is strictly before target_date:
#   [target_date - HISTORY_DAYS, target_date)
#
# ============================================================

from pathlib import Path
import json
import pandas as pd
import numpy as np

# ============================================================
# CONFIG
# ============================================================

DATA_PATH = Path(r"C:/IDEAL_Programming/processed/final/IDEAL_final_hourly_dataset.csv")

OUT_DIR = Path(r"C:/IDEAL_Programming/New_Evaluation/SEASONAL")
OUT_DIR.mkdir(parents=True, exist_ok=True)

TIME_COL = "timestamp"
HOME_COL = "home_id"
TARGET_COL = "consumption_Wh"

HISTORY_DAYS = 7
EXPECTED_HISTORY_HOURS = HISTORY_DAYS * 24
EXPECTED_TARGET_HOURS = 24

SAVE_ENCODING = "utf-8-sig"

# Seasonal historical case studies
# These dates are historical/train-period case studies, not independent test-set evaluation.
SEASONAL_CASES = [
    {
        "season": "summer",
        "home_id": "home100",
        "target_date": "2017-08-09",
        "history_stability_score": 0.869969,
        "history_stability_category": "Very High",
        "recommended_max_alpha": 0.40,
        "recommended_max_alpha_range": "0.35–0.50",
    },
    {
        "season": "autumn",
        "home_id": "home72",
        "target_date": "2017-09-27",
        "history_stability_score": 0.959708,
        "history_stability_category": "Excellent",
        "recommended_max_alpha": 0.60,
        "recommended_max_alpha_range": "0.55–0.65",
    },
    {
        "season": "winter",
        "home_id": "home268",
        "target_date": "2018-01-11",
        "history_stability_score": 0.122368,
        "history_stability_category": "Low",
        "recommended_max_alpha": 0.05,
        "recommended_max_alpha_range": "0.00–0.10",
    },
    {
        "season": "spring",
        "home_id": "home316",
        "target_date": "2018-04-23",
        "history_stability_score": 0.754679,
        "history_stability_category": "High",
        "recommended_max_alpha": 0.25,
        "recommended_max_alpha_range": "0.20–0.30",
    },
]

STATIC_COLS = [
    "hometype",
    "residents",
    "urban_rural_class",
    "total_floor_area_m2",
    "num_electric_appliances",
]

DAY_NAMES_GR = {
    0: "Δευτέρα",
    1: "Τρίτη",
    2: "Τετάρτη",
    3: "Πέμπτη",
    4: "Παρασκευή",
    5: "Σάββατο",
    6: "Κυριακή",
}


# ============================================================
# HELPERS
# ============================================================

def ensure_hourly_unique(df: pd.DataFrame) -> pd.DataFrame:
    """Ensure one row per home/timestamp.

    If duplicates exist, average consumption_Wh to avoid double counting.
    """
    tmp = df[[HOME_COL, TIME_COL, TARGET_COL]].copy()
    tmp[TARGET_COL] = pd.to_numeric(tmp[TARGET_COL], errors="coerce")
    tmp = tmp.dropna(subset=[HOME_COL, TIME_COL, TARGET_COL])

    tmp = (
        tmp
        .groupby([HOME_COL, TIME_COL], as_index=False)
        .agg({TARGET_COL: "mean"})
    )

    return tmp


def mode_or_nan(s):
    s = s.dropna()
    if s.empty:
        return np.nan
    m = s.mode()
    return m.iloc[0] if len(m) else s.iloc[0]


def make_complete_hour_index(start_ts: pd.Timestamp, end_exclusive: pd.Timestamp):
    return pd.date_range(
        start_ts,
        end_exclusive - pd.Timedelta(hours=1),
        freq="h"
    )


def add_missing_hour_check(data: pd.DataFrame, start_ts: pd.Timestamp, end_exclusive: pd.Timestamp):
    expected_index = make_complete_hour_index(start_ts, end_exclusive)
    actual_index = pd.DatetimeIndex(pd.to_datetime(data[TIME_COL]).drop_duplicates().sort_values())

    missing = expected_index.difference(actual_index)
    extra = actual_index.difference(expected_index)

    return {
        "expected_hours": len(expected_index),
        "actual_rows": len(data),
        "unique_timestamps": len(actual_index),
        "missing_hours_count": len(missing),
        "extra_hours_count": len(extra),
        "missing_hours": [str(x) for x in missing[:50]],
        "extra_hours": [str(x) for x in extra[:50]],
    }


def safe_folder_name(season: str, home_id: str, target_date: str) -> str:
    return f"{season}_{home_id}_{target_date}".replace("-", "")


def extract_static_metadata(raw_df: pd.DataFrame, home_id: str) -> dict:
    hdf = raw_df[raw_df[HOME_COL] == home_id].copy()

    row = {"home_id": home_id}

    for col in STATIC_COLS:
        if col not in hdf.columns:
            row[col] = np.nan
            continue

        if pd.api.types.is_numeric_dtype(hdf[col]):
            row[col] = float(pd.to_numeric(hdf[col], errors="coerce").median())
        else:
            row[col] = mode_or_nan(hdf[col])

    return row


def json_safe_dict(d: dict) -> dict:
    out = {}

    for k, v in d.items():
        if isinstance(v, pd.Timestamp):
            out[k] = str(v)
        elif isinstance(v, (np.integer, np.int64)):
            out[k] = int(v)
        elif isinstance(v, (np.floating, np.float64)):
            out[k] = None if pd.isna(v) else float(v)
        elif pd.isna(v) if not isinstance(v, (list, dict, str, bool, type(None))) else False:
            out[k] = None
        else:
            out[k] = v

    return out


# ============================================================
# LOAD DATA
# ============================================================

print("=" * 120)
print("Loading IDEAL final hourly dataset")
print("=" * 120)

raw_df = pd.read_csv(DATA_PATH, parse_dates=[TIME_COL], low_memory=False)
df = ensure_hourly_unique(raw_df)

print("Rows:", len(df))
print("Homes:", df[HOME_COL].nunique())
print("Date range:", df[TIME_COL].min(), "to", df[TIME_COL].max())
print("Output folder:", OUT_DIR)

summary_rows = []


# ============================================================
# CREATE FILES PER SEASONAL CASE
# ============================================================

for case in SEASONAL_CASES:
    season = case["season"]
    home_id = case["home_id"]
    target_date = pd.Timestamp(case["target_date"]).normalize()

    target_start = target_date
    target_end = target_start + pd.Timedelta(days=1)

    history_start = target_start - pd.Timedelta(days=HISTORY_DAYS)
    history_end = target_start

    day_of_week = target_start.weekday()
    day_name_gr = DAY_NAMES_GR[day_of_week]
    is_weekend = int(day_of_week >= 5)

    case_folder = safe_folder_name(season, home_id, target_start.date().isoformat())
    case_dir = OUT_DIR / case_folder
    case_dir.mkdir(parents=True, exist_ok=True)

    home_df = df[df[HOME_COL] == home_id].copy()

    if home_df.empty:
        print(f"[WARN] {home_id}: no rows found in dataset.")
        continue

    history_df = home_df[
        (home_df[TIME_COL] >= history_start) &
        (home_df[TIME_COL] < history_end)
    ].copy()

    target_df = home_df[
        (home_df[TIME_COL] >= target_start) &
        (home_df[TIME_COL] < target_end)
    ].copy()

    history_df = history_df.sort_values(TIME_COL).reset_index(drop=True)
    target_df = target_df.sort_values(TIME_COL).reset_index(drop=True)

    history_check = add_missing_hour_check(history_df, history_start, history_end)
    target_check = add_missing_hour_check(target_df, target_start, target_end)

    history_complete = (
        len(history_df) == EXPECTED_HISTORY_HOURS
        and history_check["missing_hours_count"] == 0
    )

    target_complete = (
        len(target_df) == EXPECTED_TARGET_HOURS
        and target_check["missing_hours_count"] == 0
    )

    # UI/API-ready history file: only required columns
    history_out = history_df[[TIME_COL, TARGET_COL]].copy()
    history_out[TIME_COL] = history_out[TIME_COL].dt.strftime("%Y-%m-%d %H:%M:%S")

    # Actual target day for later evaluation
    target_out = target_df[[TIME_COL, TARGET_COL]].copy()
    target_out[TIME_COL] = target_out[TIME_COL].dt.strftime("%Y-%m-%d %H:%M:%S")

    history_file = case_dir / "history_store.csv"
    target_file = case_dir / "actual_target_day.csv"

    history_out.to_csv(history_file, index=False, encoding=SAVE_ENCODING)
    target_out.to_csv(target_file, index=False, encoding=SAVE_ENCODING)

    # Save static metadata
    static_meta = extract_static_metadata(raw_df, home_id)

    static_csv = case_dir / "static_metadata.csv"
    pd.DataFrame([static_meta]).to_csv(static_csv, index=False, encoding=SAVE_ENCODING)

    static_json = case_dir / "static_metadata.json"
    with open(static_json, "w", encoding="utf-8") as f:
        json.dump(json_safe_dict(static_meta), f, ensure_ascii=False, indent=2)

    metadata = {
        "season": season,
        "home_id": home_id,
        "target_date": target_start.date().isoformat(),
        "day_name_gr": day_name_gr,
        "is_weekend": is_weekend,

        "history_days": HISTORY_DAYS,
        "history_start": str(history_start),
        "history_end_exclusive": str(history_end),
        "target_start": str(target_start),
        "target_end_exclusive": str(target_end),

        "history_file": str(history_file),
        "target_actual_file": str(target_file),
        "static_metadata_csv": str(static_csv),
        "static_metadata_json": str(static_json),

        "history_expected_hours": EXPECTED_HISTORY_HOURS,
        "history_rows": int(len(history_out)),
        "history_unique_timestamps": int(history_check["unique_timestamps"]),
        "history_missing_hours_count": int(history_check["missing_hours_count"]),
        "history_complete": bool(history_complete),

        "target_expected_hours": EXPECTED_TARGET_HOURS,
        "target_rows": int(len(target_out)),
        "target_unique_timestamps": int(target_check["unique_timestamps"]),
        "target_missing_hours_count": int(target_check["missing_hours_count"]),
        "target_complete": bool(target_complete),

        "history_total_kWh": float(history_df[TARGET_COL].sum() / 1000.0) if len(history_df) else np.nan,
        "history_mean_Wh": float(history_df[TARGET_COL].mean()) if len(history_df) else np.nan,
        "history_median_Wh": float(history_df[TARGET_COL].median()) if len(history_df) else np.nan,

        "target_total_kWh": float(target_df[TARGET_COL].sum() / 1000.0) if len(target_df) else np.nan,
        "target_mean_Wh": float(target_df[TARGET_COL].mean()) if len(target_df) else np.nan,
        "target_median_Wh": float(target_df[TARGET_COL].median()) if len(target_df) else np.nan,

        "history_stability_score": case.get("history_stability_score"),
        "history_stability_category": case.get("history_stability_category"),
        "recommended_max_alpha": case.get("recommended_max_alpha"),
        "recommended_max_alpha_range": case.get("recommended_max_alpha_range"),

        "hometype": static_meta.get("hometype"),
        "residents": static_meta.get("residents"),
        "urban_rural_class": static_meta.get("urban_rural_class"),
        "total_floor_area_m2": static_meta.get("total_floor_area_m2"),
        "num_electric_appliances": static_meta.get("num_electric_appliances"),

        "history_missing_hours_preview": history_check["missing_hours"],
        "target_missing_hours_preview": target_check["missing_hours"],

        "methodological_note": (
            "This is a historical seasonal case study. "
            "The selected date may belong to the training period and should not be treated "
            "as independent test-set evaluation."
        ),
    }

    metadata_file = case_dir / "case_metadata.json"
    with open(metadata_file, "w", encoding="utf-8") as f:
        json.dump(json_safe_dict(metadata), f, ensure_ascii=False, indent=2)

    summary_rows.append({
        "season": season,
        "case_folder": case_folder,
        "home_id": home_id,
        "target_date": target_start.date().isoformat(),
        "day_name_gr": day_name_gr,
        "is_weekend": is_weekend,

        "hometype": static_meta.get("hometype"),
        "residents": static_meta.get("residents"),
        "urban_rural_class": static_meta.get("urban_rural_class"),
        "total_floor_area_m2": static_meta.get("total_floor_area_m2"),
        "num_electric_appliances": static_meta.get("num_electric_appliances"),

        "history_start": history_start,
        "history_end_exclusive": history_end,
        "history_rows": len(history_out),
        "history_expected_hours": EXPECTED_HISTORY_HOURS,
        "history_missing_hours": history_check["missing_hours_count"],
        "history_complete": history_complete,

        "target_rows": len(target_out),
        "target_expected_hours": EXPECTED_TARGET_HOURS,
        "target_missing_hours": target_check["missing_hours_count"],
        "target_complete": target_complete,

        "history_total_kWh": metadata["history_total_kWh"],
        "target_total_kWh": metadata["target_total_kWh"],

        "history_stability_score": case.get("history_stability_score"),
        "history_stability_category": case.get("history_stability_category"),
        "recommended_max_alpha": case.get("recommended_max_alpha"),
        "recommended_max_alpha_range": case.get("recommended_max_alpha_range"),

        "history_file": str(history_file),
        "target_actual_file": str(target_file),
        "static_metadata_csv": str(static_csv),
        "metadata_file": str(metadata_file),
    })

    print("=" * 120)
    print(f"{season.upper()} | {home_id} | target_date={target_start.date()} | {day_name_gr}")
    print("=" * 120)
    print("Folder:", case_dir)
    print("History:", history_start, "to", history_end, "| rows:", len(history_out), "/", EXPECTED_HISTORY_HOURS)
    print("Target: ", target_start, "to", target_end, "| rows:", len(target_out), "/", EXPECTED_TARGET_HOURS)
    print("History complete:", history_complete)
    print("Target complete: ", target_complete)
    print("Stability:", case.get("history_stability_category"), "| score:", case.get("history_stability_score"))
    print("Recommended max_alpha:", case.get("recommended_max_alpha"))
    print("Saved history:", history_file)
    print("Saved actual target:", target_file)


# ============================================================
# SAVE GLOBAL SUMMARY
# ============================================================

summary = pd.DataFrame(summary_rows)

summary_file = OUT_DIR / "seasonal_evaluation_cases_summary.csv"
summary.to_csv(summary_file, index=False, encoding=SAVE_ENCODING)

cases_file = OUT_DIR / "selected_seasonal_case_study_homes_and_dates.csv"
pd.DataFrame(SEASONAL_CASES).to_csv(cases_file, index=False, encoding=SAVE_ENCODING)

print("=" * 120)
print("GLOBAL SEASONAL SUMMARY")
print("=" * 120)

print(
    summary[
        [
            "season",
            "case_folder",
            "home_id",
            "target_date",
            "day_name_gr",
            "hometype",
            "residents",
            "total_floor_area_m2",
            "num_electric_appliances",
            "history_rows",
            "history_expected_hours",
            "history_missing_hours",
            "history_complete",
            "target_rows",
            "target_expected_hours",
            "target_missing_hours",
            "target_complete",
            "history_stability_score",
            "history_stability_category",
            "recommended_max_alpha",
            "history_total_kWh",
            "target_total_kWh",
        ]
    ].to_string(index=False)
)

print("=" * 120)
print("Saved")
print("=" * 120)
print("Summary:", summary_file)
print("Cases:", cases_file)
print("Root output folder:", OUT_DIR)

Loading IDEAL final hourly dataset
Rows: 1529350
Homes: 254
Date range: 2016-08-10 10:00:00 to 2018-06-30 23:00:00
Output folder: C:\IDEAL_Programming\New_Evaluation\SEASONAL
SUMMER | home100 | target_date=2017-08-09 | Τετάρτη
Folder: C:\IDEAL_Programming\New_Evaluation\SEASONAL\summer_home100_20170809
History: 2017-08-02 00:00:00 to 2017-08-09 00:00:00 | rows: 168 / 168
Target:  2017-08-09 00:00:00 to 2017-08-10 00:00:00 | rows: 24 / 24
History complete: True
Target complete:  True
Stability: Very High | score: 0.869969
Recommended max_alpha: 0.4
Saved history: C:\IDEAL_Programming\New_Evaluation\SEASONAL\summer_home100_20170809\history_store.csv
Saved actual target: C:\IDEAL_Programming\New_Evaluation\SEASONAL\summer_home100_20170809\actual_target_day.csv
AUTUMN | home72 | target_date=2017-09-27 | Τετάρτη
Folder: C:\IDEAL_Programming\New_Evaluation\SEASONAL\autumn_home72_20170927
History: 2017-09-20 00:00:00 to 2017-09-27 00:00:00 | rows: 168 / 168
Target:  2017-09-27 00:00:00 to 201